In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# ── Config ────────────────────────────────────────────────────────────────────
DATASET_PATH = "/kaggle/input/datasets/nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
SEED         = 42
CLASSES      = ["Cyst", "Tumor", "Stone", "Normal"]
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)}

# ── Load paths + labels ───────────────────────────────────────────────────────
all_paths, all_labels = [], []
for cls in CLASSES:
    cls_folder = os.path.join(DATASET_PATH, cls)
    for fname in os.listdir(cls_folder):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_paths.append(os.path.join(cls_folder, fname))
            all_labels.append(CLASS_TO_IDX[cls])

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)

# ── Splits ────────────────────────────────────────────────────────────────────
X_train, X_temp, y_train, y_temp = train_test_split(
    all_paths, all_labels, test_size=0.30, stratify=all_labels, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

# ── Class weights ─────────────────────────────────────────────────────────────
class_weights_array = compute_class_weight(
    class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights_array))

# ── Preprocessing: feed RAW [0,255] — EfficientNet handles its own scaling ────
def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)   # ✅ NO /255 — EfficientNetB0 does it internally
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.1 * 255)  # scale to [0,255] range
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = tf.image.rot90(img, k=tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32))
    img = tf.clip_by_value(img, 0.0, 255.0)                     # clip to valid range
    return img, label

def build_dataset(paths, labels, augment_data=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = build_dataset(X_train, y_train, augment_data=True,  shuffle=True)
val_ds   = build_dataset(X_val,   y_val,   augment_data=False, shuffle=False)
test_ds  = build_dataset(X_test,  y_test,  augment_data=False, shuffle=False)

# ── Verify pixel range is now [0, 255] ────────────────────────────────────────
for imgs, labels in train_ds.take(1):
    print(f"Batch shape : {imgs.shape}")
    print(f"Pixel range : [{imgs.numpy().min():.1f}, {imgs.numpy().max():.1f}]  ✅ should be ~[0, 255]")

# ── Build Model ───────────────────────────────────────────────────────────────
tf.keras.backend.clear_session()

base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
    # include_preprocessing=True by default → expects [0,255] ✅
)
base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(4, activation='softmax')(x)
model = Model(inputs, outputs)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"\nTrainable params : {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")
print(f"Frozen params    : {sum([tf.size(w).numpy() for w in model.non_trainable_weights]):,}")

# ── Phase 1 Training ──────────────────────────────────────────────────────────
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

callbacks_phase1 = [
    ModelCheckpoint(
        filepath="/kaggle/working/checkpoints/best_phase1.keras",
        monitor="val_accuracy", save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor="val_accuracy", patience=5,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss", factor=0.5,
        patience=3, min_lr=1e-6, verbose=1
    )
]

print("\n" + "="*50)
print("PHASE 1: Training head only (base frozen)")
print("="*50)

history_phase1 = model.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    class_weight=class_weights,
    callbacks=callbacks_phase1,
    verbose=1
)

best_val_acc = max(history_phase1.history['val_accuracy'])
print(f"\nBest Val Accuracy (Phase 1): {best_val_acc:.4f}")

In [ ]:
# ── Unfreeze top 30 layers of base model ─────────────────────────────────────
base_model.trainable = True

# Freeze all except last 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Verify
trainable   = sum([tf.size(w).numpy() for w in model.trainable_weights])
non_trainable = sum([tf.size(w).numpy() for w in model.non_trainable_weights])
print(f"Trainable params   : {trainable:,}")
print(f"Non-trainable params: {non_trainable:,}")

# ── Recompile with very low learning rate ─────────────────────────────────────
model.compile(
    optimizer=Adam(learning_rate=1e-5),   # 100x smaller than Phase 1
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ── Callbacks ─────────────────────────────────────────────────────────────────
callbacks_phase2 = [
    ModelCheckpoint(
        filepath="/kaggle/working/checkpoints/best_phase2.keras",
        monitor="val_accuracy", save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor="val_accuracy", patience=6,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss", factor=0.5,
        patience=3, min_lr=1e-7, verbose=1
    )
]

# ── Phase 2 Training ──────────────────────────────────────────────────────────
print("="*50)
print("PHASE 2: Fine-tuning top 30 layers")
print("="*50)

history_phase2 = model.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    class_weight=class_weights,
    callbacks=callbacks_phase2,
    verbose=1
)

# ── Plot both phases ──────────────────────────────────────────────────────────
def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    ax1.plot(history.history['accuracy'],     label='Train Acc')
    ax1.plot(history.history['val_accuracy'], label='Val Acc')
    ax1.set_title('Accuracy'); ax1.legend(); ax1.set_xlabel('Epoch')
    ax2.plot(history.history['loss'],     label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Val Loss')
    ax2.set_title('Loss'); ax2.legend(); ax2.set_xlabel('Epoch')
    plt.tight_layout()
    plt.show()

plot_history(history_phase2, "Phase 2 — Fine-tuning")

best_val_acc_p2 = max(history_phase2.history['val_accuracy'])
print(f"\nBest Val Accuracy (Phase 1) : 0.9914")
print(f"Best Val Accuracy (Phase 2) : {best_val_acc_p2:.4f}")
print(f"Improvement                 : {(best_val_acc_p2 - 0.9914)*100:+.2f}%")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ── Test set evaluation ───────────────────────────────────────────────────────
print("="*50)
print("FINAL EVALUATION ON TEST SET")
print("="*50)

test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f"\nTest Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Test Loss     : {test_loss:.4f}")

# ── Get predictions ───────────────────────────────────────────────────────────
y_pred_probs = model.predict(test_ds, verbose=1)
y_pred       = np.argmax(y_pred_probs, axis=1)

# ── Classification Report ─────────────────────────────────────────────────────
print("\n--- Classification Report ---")
print(classification_report(
    y_test, y_pred,
    target_names=CLASSES,
    digits=4
))

# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASSES,
    yticklabels=CLASSES,
    linewidths=0.5
)
plt.title('Confusion Matrix — Test Set', fontsize=14, fontweight='bold')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# ── Per-class accuracy ────────────────────────────────────────────────────────
print("\n--- Per-Class Accuracy ---")
for i, cls in enumerate(CLASSES):
    cls_mask    = y_test == i
    cls_correct = np.sum((y_pred == i) & cls_mask)
    cls_total   = np.sum(cls_mask)
    print(f"  {cls:8s}: {cls_correct}/{cls_total} = {cls_correct/cls_total*100:.2f}%")

In [ ]:
# ── Save final model ──────────────────────────────────────────────────────────
model.save("/kaggle/working/kidney_classifier_final.keras")
print("✅ Model saved to /kaggle/working/kidney_classifier_final.keras")

# ── Visualize predictions on random test samples ──────────────────────────────
import random

# Load some raw test images for display
def load_display_image(path):
    img = Image.open(path).resize(IMG_SIZE)
    return np.array(img)

# Pick 12 random test samples
random.seed(99)
sample_indices = random.sample(range(len(X_test)), 12)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle("Model Predictions on Test Samples", fontsize=16, fontweight='bold')
axes = axes.flatten()

for i, idx in enumerate(sample_indices):
    img_display = load_display_image(X_test[idx])
    
    # Predict
    img_tensor = tf.cast(tf.image.resize(
        tf.io.decode_jpeg(tf.io.read_file(X_test[idx]), channels=3),
        IMG_SIZE), tf.float32)
    pred_probs  = model.predict(tf.expand_dims(img_tensor, 0), verbose=0)[0]
    pred_class  = CLASSES[np.argmax(pred_probs)]
    true_class  = CLASSES[y_test[idx]]
    confidence  = np.max(pred_probs) * 100
    correct     = pred_class == true_class

    axes[i].imshow(img_display, cmap='gray')
    axes[i].axis('off')
    color = 'green' if correct else 'red'
    axes[i].set_title(
        f"True: {true_class}\nPred: {pred_class} ({confidence:.1f}%)",
        color=color, fontsize=10, fontweight='bold'
    )

plt.tight_layout()
plt.show()

# ── Final summary ─────────────────────────────────────────────────────────────
print("\n" + "="*50)
print("🏆 FINAL MODEL SUMMARY")
print("="*50)
print(f"  Architecture  : EfficientNetB0 + Custom Head")
print(f"  Classes       : {CLASSES}")
print(f"  Total images  : 12,446")
print(f"  Train/Val/Test: 70% / 15% / 15%")
print(f"  Phase 1 Acc   : 99.14% (frozen base)")
print(f"  Phase 2 Acc   : 99.57% (fine-tuned)")
print(f"  Test Accuracy : 99.30%")
print(f"  Test F1 Score : 99.22% (macro avg)")
print(f"  Stone Recall  : 100.00% ✨")
print(f"  Model saved   : /kaggle/working/kidney_classifier_final.keras")
print("="*50)